# ChemicalEntity ↔ Protein Relation-Wise Merge

Merges Chemical–Protein triples from CKG (×2), CrossBAR (×2), DtiNet, STITCH,
and MetaboGlue (×3); resolves chemical names via PubChem/DrugBank and protein
names via UniProt; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_PROTEIN/ALL_CHEMICALENTITY_PROTEIN.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical — PubChem and DrugBank

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
# del Pubchem
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


In [3]:
Pubchem_IUPAC_CID_Dict_rev = {
    v: k for k, v in Pubchem_IUPAC_CID_Dict.items()
}

In [4]:
Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))
print(f"DrugBank dict: {len(Drugbank_dict):,} entries")
Drugbank

DrugBank dict: 16,575 entries


,Unnamed: 0,drugbank_id,name,type,groups,atc_codes,categories,inchikey,inchi,SMILES,description,Pubchem_ID
0,0,DB00001,Lepirudin,biotech,approved|withdrawn,B01AE02,"Amino Acids, Peptides, and Proteins|Anticoagul...",NaN,NaN,NaN,Lepirudin is a recombinant hirudin formed by 6...,NaN
1,1,DB00002,Cetuximab,biotech,approved,L01FE01,"Amino Acids, Peptides, and Proteins|Antibodies...",NaN,NaN,NaN,Cetuximab is a recombinant chimeric human/mous...,NaN
2,2,DB00003,Dornase alfa,biotech,approved,R05CB13,"Amino Acids, Peptides, and Proteins|Cough and ...",NaN,NaN,NaN,Dornase alfa is a biosynthetic form of human d...,NaN
3,3,DB00004,Denileukin diftitox,biotech,approved|investigational,L01XX29,"ADP Ribose Transferases|Amino Acids, Peptides,...",NaN,NaN,NaN,A recombinant DNA-derived cytotoxic protein co...,NaN
4,4,DB00005,Etanercept,biotech,approved|investigational,L04AB01,"Agents reducing cytokine levels|Amino Acids, P...",NaN,NaN,NaN,Dimeric fusion protein consisting of the extra...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
16570,16570,DB18707,AVL-3288,small molecule,investigational,NaN,NaN,VMAKIACTLSBBIY-BOPFTXTBSA-N,InChI=1S/C19H15Cl2N3O2/c1-12-10-18(26-24-12)17...,CC1=NOC(=C1)C(=C\NC1=CC=C(Cl)C=C1)\C(=O)NC1=CC...,NaN,NaN
16571,16571,DB18708,Alogabat,small molecule,investigational,NaN,NaN,ACZCJTHHWMBFKC-UHFFFAOYSA-N,InChI=1S/C21H23N5O4/c1-13-3-4-15(11-22-13)20-1...,CC1=C(COC2=CC=C(N=N2)C(=O)NC2CCOCC2)C(=NO1)C1=...,NaN,NaN
16572,16572,DB18709,Ropsacitinib,small molecule,investigational,NaN,NaN,XPLZTJWZDBFWDE-OYOVHJISSA-N,InChI=1S/C20H17N9/c1-27-11-15(9-24-27)17-13-28...,CN1C=C(C=N1)C1=CN2N=CC=C2C(=N1)C1=CN(N=C1)[C@@...,NaN,NaN
16573,16573,DB18710,ABBV-706,biotech,investigational,NaN,NaN,NaN,NaN,NaN,ABBV-706 is an antibody-drug conjugate that is...,NaN


In [10]:
HMDB = pd.read_csv(DB_DIR + 'hmdb/HMDB_withname_pubchem.csv')
HMDB['PUBCHEM_ID'] = HMDB['PUBCHEM_ID'].apply(
    lambda x: str(int(x)) if pd.notna(x) else x
)
HMDB

,accession,name,description,synonym,iupac_name,smiles,inchi,Pubchem,PUBCHEM_ID
0,HMDB0000001,1-methylhistidine,"1-Methylhistidine, also known as 1-MHis or 1MH...",(2S)-2-Amino-3-(1-methyl-1H-imidazol-4-yl)prop...,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,CN1C=NC(C[C@H](N)C(O)=O)=C1,InChI=1S/C7H11N3O2/c1-10-3-5(9-4-10)2-6(8)7(11...,92105.0,92105
1,HMDB0000002,"1,3-diaminopropane","1,3-Diaminopropane, also known as DAP or trime...","1,3-Propanediamine, 1,3-Propylenediamine, Prop...","propane-1,3-diamine",NCCCN,InChI=1S/C3H10N2/c4-2-1-3-5/h1-5H2,157742609.0,428
2,HMDB0000005,2-ketobutyric acid,"2-Ketobutyric acid, also known as alpha-ketobu...","2-Ketobutanoic acid, 2-Oxobutyric acid, 3-Meth...",2-oxobutanoic acid,CCC(=O)C(O)=O,"InChI=1S/C4H6O3/c1-2-3(5)4(6)7/h2H2,1H3,(H,6,7)",58.0,58
3,HMDB0000008,2-hydroxybutyric acid,"2-Hydroxybutyric acid (CAS: 600-15-7), also kn...","(S)-2-Hydroxybutanoic acid, 2-Hydroxybutyrate,...",(2S)-2-hydroxybutanoic acid,CC[C@H](O)C(O)=O,"InChI=1S/C4H8O3/c1-2-3(5)4(6)7/h3,5H,2H2,1H3,(...",157608701.0,11266
4,HMDB0000010,2-methoxyestrone,2-Methoxyestrone (or 2-ME1) belongs to the cla...,"2-(8S,9S,13S,14S)-3-Hydroxy-2-methoxy-13-methy...","(1S,10R,11S,15S)-5-hydroxy-4-methoxy-15-methyl...",[H][C@@]12CCC(=O)[C@@]1(C)CC[C@]1([H])C3=C(CC[...,InChI=1S/C19H24O3/c1-19-8-7-12-13(15(19)5-6-18...,NaN,440624
...,...,...,...,...,...,...,...,...,...
217915,HMDB0304947,nordeoxycholic acid,NaN,NaN,"3-{5,16-dihydroxy-2,15-dimethyltetracyclo[8.7....",CC(CC(O)=O)C1CCC2C3CCC4CC(O)CCC4(C)C3CC(O)C12C,InChI=1S/C23H38O4/c1-13(10-21(26)27)17-6-7-18-...,193905.0,193905
217916,HMDB0304950,3-oxo-5beta-cholanoic acid,NaN,3-Ketocholanate,"4-{2,15-dimethyl-5-oxotetracyclo[8.7.0.0^{2,7}...",CC(CCC(O)=O)C1CCC2C3CCC4CC(=O)CCC4(C)C3CCC12C,InChI=1S/C24H38O3/c1-15(4-9-22(26)27)19-7-8-20...,NaN,5283906
217917,HMDB0304951,glycerol 1-myristate,NaN,"1-Myristoyl glycerol, 1-Tetradecanoylglycerol,...","2,3-dihydroxypropyl tetradecanoate",CCCCCCCCCCCCCC(=O)OCC(O)CO,InChI=1S/C17H34O4/c1-2-3-4-5-6-7-8-9-10-11-12-...,7905.0,79050
217918,HMDB0304953,o-phenolsulfonic acid,NaN,"O-Hydroxybenzenesulfonic acid, O-Phenolsulfoni...",2-hydroxybenzene-1-sulfonic acid,OC1=CC=CC=C1S(O)(=O)=O,"InChI=1S/C6H6O4S/c7-5-3-1-2-4-6(5)11(8,9)10/h1...",NaN,11867


In [11]:
HMDB_PC_dict = dict(zip(HMDB['accession'],HMDB['PUBCHEM_ID']))


### 1b. Protein — UniProt

In [5]:
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} entries")

UniProt RecName dict: 794,151 entries


## 2. Load KG Sources

### 2a. CKG (×2)

In [6]:
CKG_Chemical_Protein_1 = pd.read_csv(PROC_DIR + 'CKG/File_38_Drug_Protein_CKG.csv')
CKG_Chemical_Protein_1.columns = CKG_Chemical_Protein_1.columns.str.lower()
print(f"CKG 1:     {len(CKG_Chemical_Protein_1):,} rows")

CKG_Chemical_Protein_1['relation'] = 'ChemicalEntity_Protein'
CKG_Chemical_Protein_1['head_type'] = 'ChemicalEntity'
CKG_Chemical_Protein_1['tail_type'] = 'Protein'
CKG_Chemical_Protein_1['kg_type'] = 'Generalised'
CKG_Chemical_Protein_1['kg_source'] = 'CKG'
CKG_Chemical_Protein_1['species'] = 'Homo species'

CKG_Chemical_Protein_1['head_detail_name'] = CKG_Chemical_Protein_1['head_smiles'].fillna(CKG_Chemical_Protein_1['head_iupac'])
CKG_Chemical_Protein_1['tail_detail_name'] = CKG_Chemical_Protein_1['tail_alt_name']
CKG_Chemical_Protein_1

CKG 1:     832,817 rows


,head,relation,tail,head_type,relation_type,tail_type,source,kg_source,head_alt_ids,head_iupac,head_smiles,tail_alt_name,head_id_is,tail_id_is,evidence,kg_type,species,head_detail_name,tail_detail_name
0,DB06439,ChemicalEntity_Protein,Q16283,ChemicalEntity,association,Protein,STITCH,CKG,DB06439,Tyloxapol,NaN,Lipoprotein lipase,Drugbank,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,Tyloxapol,Lipoprotein lipase
1,DB06439,ChemicalEntity_Protein,B2R5T9,ChemicalEntity,association,Protein,STITCH,CKG,DB06439,Tyloxapol,NaN,Lipoprotein lipase,Drugbank,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,Tyloxapol,Lipoprotein lipase
2,DB06439,ChemicalEntity_Protein,P06858,ChemicalEntity,association,Protein,STITCH,CKG,DB06439,Tyloxapol,NaN,Lipoprotein lipase,Drugbank,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,Tyloxapol,Lipoprotein lipase
3,DB06439,ChemicalEntity_Protein,Q96FC4,ChemicalEntity,association,Protein,STITCH,CKG,DB06439,Tyloxapol,NaN,Lipoprotein lipase,Drugbank,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,Tyloxapol,Lipoprotein lipase
4,DB06439,ChemicalEntity_Protein,Q16282,ChemicalEntity,association,Protein,STITCH,CKG,DB06439,Tyloxapol,NaN,Lipoprotein lipase,Drugbank,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,Tyloxapol,Lipoprotein lipase
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832812,7045767,ChemicalEntity_Protein,Q53T69,ChemicalEntity,association,Protein,STITCH,CKG,DB08842,CC(=O)O[C@H](CC(=O)[O-])C[N+](C)(C)C,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Long chain 3-hydroxyacyl-CoA dehydrogenase,Pubchem,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Long chain 3-hydroxyacyl-CoA dehydrogenase
832813,7045767,ChemicalEntity_Protein,Q16679,ChemicalEntity,association,Protein,STITCH,CKG,DB08842,CC(=O)O[C@H](CC(=O)[O-])C[N+](C)(C)C,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Long chain 3-hydroxyacyl-CoA dehydrogenase,Pubchem,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Long chain 3-hydroxyacyl-CoA dehydrogenase
832814,7045767,ChemicalEntity_Protein,B2R7L4,ChemicalEntity,association,Protein,STITCH,CKG,DB08842,CC(=O)O[C@H](CC(=O)[O-])C[N+](C)(C)C,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Long chain 3-hydroxyacyl-CoA dehydrogenase,Pubchem,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Long chain 3-hydroxyacyl-CoA dehydrogenase
832815,7045767,ChemicalEntity_Protein,A1XKG3,ChemicalEntity,association,Protein,STITCH,CKG,DB08842,CC(=O)O[C@H](CC(=O)[O-])C[N+](C)(C)C,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Cyclin-dependent kinase 5 {ECO:0000312|HGNC:HG...,Pubchem,UniprotID,"experimental,prediction,database,textmining,score",Generalised,Homo species,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Cyclin-dependent kinase 5 {ECO:0000312|HGNC:HG...


In [7]:
CKG_Chemical_Protein_2 = pd.read_csv(PROC_DIR + 'CKG/File_39_Drug_Protein_CKG.csv')
CKG_Chemical_Protein_2.columns = CKG_Chemical_Protein_2.columns.str.lower()

print(f"CKG 2:     {len(CKG_Chemical_Protein_2):,} rows")

CKG_Chemical_Protein_2['relation'] = 'ChemicalEntity_Protein'
CKG_Chemical_Protein_2['head_type'] = 'ChemicalEntity'
CKG_Chemical_Protein_2['tail_type'] = 'Protein'
CKG_Chemical_Protein_2['kg_type'] = 'Generalised'
CKG_Chemical_Protein_2['kg_source'] = 'CKG'
CKG_Chemical_Protein_2['species'] = 'Homo species'

CKG_Chemical_Protein_2['head_detail_name'] = CKG_Chemical_Protein_2['head_smiles'].fillna(CKG_Chemical_Protein_2['head_iupac'])
CKG_Chemical_Protein_2['tail_detail_name'] = CKG_Chemical_Protein_2['tail_alt_name']

CKG_Chemical_Protein_2

CKG 2:     1,659,588 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_alt_ids,head_iupac,head_smiles,tail_alt_name,head_id_is,tail_id_is,kg_type,species,head_detail_name,tail_detail_name
0,6306,ChemicalEntity_Protein,Q8IYP2,ChemicalEntity,Protein,STITCH,CKG,DB00167,CC[C@H](C)[C@@H](C(=O)O)N,"(2S,3S)-2-amino-3-methylpentanoic acid",Serine protease 58,Pubchem,UniprotID,Generalised,Homo species,"(2S,3S)-2-amino-3-methylpentanoic acid",Serine protease 58
1,6306,ChemicalEntity_Protein,B3KVJ6,ChemicalEntity,Protein,STITCH,CKG,DB00167,CC[C@H](C)[C@@H](C(=O)O)N,"(2S,3S)-2-amino-3-methylpentanoic acid",Serine protease 58,Pubchem,UniprotID,Generalised,Homo species,"(2S,3S)-2-amino-3-methylpentanoic acid",Serine protease 58
2,6306,ChemicalEntity_Protein,D3DXD2,ChemicalEntity,Protein,STITCH,CKG,DB00167,CC[C@H](C)[C@@H](C(=O)O)N,"(2S,3S)-2-amino-3-methylpentanoic acid",Serine protease 58,Pubchem,UniprotID,Generalised,Homo species,"(2S,3S)-2-amino-3-methylpentanoic acid",Serine protease 58
3,99288,ChemicalEntity_Protein,Q8IYP2,ChemicalEntity,Protein,STITCH,CKG,DB01739,CC[C@@H](C)[C@@H](C(=O)O)N,"(2S,3R)-2-amino-3-methylpentanoic acid",Serine protease 58,Pubchem,UniprotID,Generalised,Homo species,"(2S,3R)-2-amino-3-methylpentanoic acid",Serine protease 58
4,99288,ChemicalEntity_Protein,B3KVJ6,ChemicalEntity,Protein,STITCH,CKG,DB01739,CC[C@@H](C)[C@@H](C(=O)O)N,"(2S,3R)-2-amino-3-methylpentanoic acid",Serine protease 58,Pubchem,UniprotID,Generalised,Homo species,"(2S,3R)-2-amino-3-methylpentanoic acid",Serine protease 58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1659583,12560,ChemicalEntity_Protein,Q6AZZ6,ChemicalEntity,Protein,STITCH,CKG,DB00199,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2,Pubchem,UniprotID,Generalised,Homo species,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2
1659584,12560,ChemicalEntity_Protein,Q13457,ChemicalEntity,Protein,STITCH,CKG,DB00199,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2,Pubchem,UniprotID,Generalised,Homo species,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2
1659585,12560,ChemicalEntity_Protein,Q4W5G7,ChemicalEntity,Protein,STITCH,CKG,DB00199,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2,Pubchem,UniprotID,Generalised,Homo species,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2
1659586,12560,ChemicalEntity_Protein,Q9UE67,ChemicalEntity,Protein,STITCH,CKG,DB00199,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2,Pubchem,UniprotID,Generalised,Homo species,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",Neuropeptide Y receptor type 2


In [12]:
CKG_Chemical_Protein_3 = pd.read_csv(PROC_DIR + 'CKG/File_32_Metabolite_Protein_CKG.csv')
CKG_Chemical_Protein_3.columns = CKG_Chemical_Protein_3.columns.str.lower()

print(f"CKG 3:     {len(CKG_Chemical_Protein_3):,} rows")

CKG_Chemical_Protein_3['relation'] = 'ChemicalEntity_Protein'
CKG_Chemical_Protein_3['head_type'] = 'ChemicalEntity'
CKG_Chemical_Protein_3['tail_type'] = 'Protein'
CKG_Chemical_Protein_3['kg_type'] = 'Generalised'
CKG_Chemical_Protein_3['kg_source'] = 'CKG'
CKG_Chemical_Protein_3['species'] = 'Homo species'

CKG_Chemical_Protein_3['head_id'] = CKG_Chemical_Protein_3['head_iupac_name'].map(Pubchem_IUPAC_CID_Dict_rev)
# CKG_Chemical_Protein_3['head_detail_name'] = CKG_Chemical_Protein_3['head_smiles'].fillna(CKG_Chemical_Protein_3['head_iupac'])
# CKG_Chemical_Protein_3['tail_detail_name'] = CKG_Chemical_Protein_3['tail_alt_name']
CKG_Chemical_Protein_3['head_id'] = CKG_Chemical_Protein_3['head'].map(HMDB_PC_dict)
CKG_Chemical_Protein_3[['head', 'head_id']] = (CKG_Chemical_Protein_3[['head_id', 'head']])
CKG_Chemical_Protein_3

CKG 3:     863,759 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_common_name,head_iupac_name,tail_alt_name,head_id_is,tail_id_is,kg_type,species,head_id
0,92105,ChemicalEntity_Protein,Q96KN2,ChemicalEntity,Protein,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,Beta-Ala-His dipeptidase {ECO:0000305},HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000001
1,92105,ChemicalEntity_Protein,O60678,ChemicalEntity,Protein,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,Protein arginine N-methyltransferase 3 {ECO:00...,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000001
2,428,ChemicalEntity_Protein,P49366,ChemicalEntity,Protein,HMDB,CKG,"1,3-diaminopropane","propane-1,3-diamine",Deoxyhypusine synthase,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000002
3,428,ChemicalEntity_Protein,P52788,ChemicalEntity,Protein,HMDB,CKG,"1,3-diaminopropane","propane-1,3-diamine",Spermine synthase {ECO:0000303|PubMed:7546290},HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000002
4,428,ChemicalEntity_Protein,P17707,ChemicalEntity,Protein,HMDB,CKG,"1,3-diaminopropane","propane-1,3-diamine",S-adenosylmethionine decarboxylase beta chain,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
863754,NaN,ChemicalEntity_Protein,E7FB98,ChemicalEntity,Protein,HMDB,CKG,vitamin k2,"2-methyl-3-(3,7,11,15-tetramethylhexadeca-2,6,...",UbiA prenyltransferase domain-containing prote...,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259856
863755,NaN,ChemicalEntity_Protein,Q9Y5Z9,ChemicalEntity,Protein,HMDB,CKG,vitamin k2,"2-methyl-3-(3,7,11,15-tetramethylhexadeca-2,6,...",UbiA prenyltransferase domain-containing prote...,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259856
863756,16131039,ChemicalEntity_Protein,P40765,ChemicalEntity,Protein,HMDB,CKG,xenin,4-{[1-({5-amino-1-[(1-{[1-({1-[(1-{[5-amino-1-...,Proxenin,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259928
863757,16131039,ChemicalEntity_Protein,P53621,ChemicalEntity,Protein,HMDB,CKG,xenin,4-{[1-({5-amino-1-[(1-{[1-({1-[(1-{[5-amino-1-...,Proxenin,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259928


In [14]:
CKG_Chemical_Protein_3 = CKG_Chemical_Protein_3[~CKG_Chemical_Protein_3['head'].isna()]
CKG_Chemical_Protein_3

,head,relation,tail,head_type,tail_type,source,kg_source,head_common_name,head_iupac_name,tail_alt_name,head_id_is,tail_id_is,kg_type,species,head_id
0,92105,ChemicalEntity_Protein,Q96KN2,ChemicalEntity,Protein,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,Beta-Ala-His dipeptidase {ECO:0000305},HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000001
1,92105,ChemicalEntity_Protein,O60678,ChemicalEntity,Protein,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,Protein arginine N-methyltransferase 3 {ECO:00...,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000001
2,428,ChemicalEntity_Protein,P49366,ChemicalEntity,Protein,HMDB,CKG,"1,3-diaminopropane","propane-1,3-diamine",Deoxyhypusine synthase,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000002
3,428,ChemicalEntity_Protein,P52788,ChemicalEntity,Protein,HMDB,CKG,"1,3-diaminopropane","propane-1,3-diamine",Spermine synthase {ECO:0000303|PubMed:7546290},HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000002
4,428,ChemicalEntity_Protein,P17707,ChemicalEntity,Protein,HMDB,CKG,"1,3-diaminopropane","propane-1,3-diamine",S-adenosylmethionine decarboxylase beta chain,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0000002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
863750,8024,ChemicalEntity_Protein,A5PLL7,ChemicalEntity,Protein,HMDB,CKG,vinyl ether,(ethenyloxy)ethene,Plasmanylethanolamine desaturase 1 {ECO:000031...,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259824
863751,8024,ChemicalEntity_Protein,Q9AE87,ChemicalEntity,Protein,HMDB,CKG,vinyl ether,(ethenyloxy)ethene,Plasmanylethanolamine desaturase {ECO:0000303|...,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259824
863756,16131039,ChemicalEntity_Protein,P40765,ChemicalEntity,Protein,HMDB,CKG,xenin,4-{[1-({5-amino-1-[(1-{[1-({1-[(1-{[5-amino-1-...,Proxenin,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259928
863757,16131039,ChemicalEntity_Protein,P53621,ChemicalEntity,Protein,HMDB,CKG,xenin,4-{[1-({5-amino-1-[(1-{[1-({1-[(1-{[5-amino-1-...,Proxenin,HMDB,Uniprot_ID,Generalised,Homo species,HMDB0259928


### 2b. CrossBAR (×2)

In [15]:
CrossBAR_Chemical_Protein_1 = pd.read_csv(PROC_DIR + 'crossbar/ChemicalEntity_Protein_Crossbar.csv')
CrossBAR_Chemical_Protein_1.columns = CrossBAR_Chemical_Protein_1.columns.str.lower()
CrossBAR_Chemical_Protein_1['head'] = CrossBAR_Chemical_Protein_1['head'].astype(str).str.replace(r'\.0$', '', regex=True)
print(f"CrossBAR 1: {len(CrossBAR_Chemical_Protein_1):,} rows")

CrossBAR_Chemical_Protein_1['kg_type'] = 'Generalised'
CrossBAR_Chemical_Protein_1['kg_source'] = 'CROssBAR'
CrossBAR_Chemical_Protein_1['species'] = 'Homo species'
CrossBAR_Chemical_Protein_1

CrossBAR 1: 15,201 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,2519,ChemicalEntity_Protein,Q13315,ChemicalEntity,Protein,CROssBAR,DB00201,ATM,Pubchem,Uniprot_ID,Generalised,Homo species
1,46937145,ChemicalEntity_Protein,P12883,ChemicalEntity,Protein,CROssBAR,DB08378,MYH7,Pubchem,Uniprot_ID,Generalised,Homo species
2,448915,ChemicalEntity_Protein,P03372,ChemicalEntity,Protein,CROssBAR,DB04471,ESR1,Pubchem,Uniprot_ID,Generalised,Homo species
3,448577,ChemicalEntity_Protein,P03372,ChemicalEntity,Protein,CROssBAR,DB03742,ESR1,Pubchem,Uniprot_ID,Generalised,Homo species
4,23642301,ChemicalEntity_Protein,P03372,ChemicalEntity,Protein,CROssBAR,DB06374,ESR1,Pubchem,Uniprot_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...
15196,135398619,ChemicalEntity_Protein,P62491,ChemicalEntity,Protein,CROssBAR,DB04315,RAB11A,Pubchem,Uniprot_ID,Generalised,Homo species
15197,1567,ChemicalEntity_Protein,Q96GD3,ChemicalEntity,Protein,CROssBAR,DB03345,SCMH1,Pubchem,Uniprot_ID,Generalised,Homo species
15198,1082,ChemicalEntity_Protein,P02749,ChemicalEntity,Protein,CROssBAR,DB05446,APOH,Pubchem,Uniprot_ID,Generalised,Homo species
15199,23978,ChemicalEntity_Protein,P02749,ChemicalEntity,Protein,CROssBAR,DB09130,APOH,Pubchem,Uniprot_ID,Generalised,Homo species


In [16]:

CrossBAR_Chemical_Protein_2 = pd.read_csv(PROC_DIR + 'crossbar/Chemical_Protein_Crossbar.csv')
CrossBAR_Chemical_Protein_2.columns = CrossBAR_Chemical_Protein_2.columns.str.lower()
CrossBAR_Chemical_Protein_2['head'] = CrossBAR_Chemical_Protein_2['head'].astype(str).str.replace(r'\.0$', '', regex=True)

CrossBAR_Chemical_Protein_2['kg_type'] = 'Generalised'
CrossBAR_Chemical_Protein_2['kg_source'] = 'CROssBAR'
CrossBAR_Chemical_Protein_2['species'] = 'Homo species'

print(f"CrossBAR 2: {len(CrossBAR_Chemical_Protein_2):,} rows")
CrossBAR_Chemical_Protein_2

CrossBAR 2: 568 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,tail_alt_id,head_id_is,tail_id_is,kg_type,species
0,1973720,ChemicalEntity_Protein,P62136,ChemicalEntity,Protein,CROssBAR,CHEMBL1533230,PPP1CA,Pubchem,Uniprot_ID,Generalised,Homo species
1,145985888,ChemicalEntity_Protein,P07711,ChemicalEntity,Protein,CROssBAR,CHEMBL4237990,CTSL,Pubchem,Uniprot_ID,Generalised,Homo species
2,44398651,ChemicalEntity_Protein,P07711,ChemicalEntity,Protein,CROssBAR,CHEMBL190121,CTSL,Pubchem,Uniprot_ID,Generalised,Homo species
3,14309405,ChemicalEntity_Protein,P04035,ChemicalEntity,Protein,CROssBAR,CHEMBL482601,HMGCR,Pubchem,Uniprot_ID,Generalised,Homo species
4,6476700,ChemicalEntity_Protein,O60341,ChemicalEntity,Protein,CROssBAR,CHEMBL226102,KDM1A,Pubchem,Uniprot_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...
563,146089,ChemicalEntity_Protein,O00748,ChemicalEntity,Protein,CROssBAR,CHEMBL89506,CES2,Pubchem,Uniprot_ID,Generalised,Homo species
564,9817358,ChemicalEntity_Protein,P22894,ChemicalEntity,Protein,CROssBAR,CHEMBL116150,MMP8,Pubchem,Uniprot_ID,Generalised,Homo species
565,70681470,ChemicalEntity_Protein,P48147,ChemicalEntity,Protein,CROssBAR,CHEMBL2024674,PREP,Pubchem,Uniprot_ID,Generalised,Homo species
566,44345896,ChemicalEntity_Protein,Q9HBH1,ChemicalEntity,Protein,CROssBAR,CHEMBL422256,PDF,Pubchem,Uniprot_ID,Generalised,Homo species


### 2c. DtiNet

In [17]:
DtiNet_Chemical_Protein = pd.read_csv(PROC_DIR + 'DTINet/Chemical_Protein_DTINet.csv')
DtiNet_Chemical_Protein.columns = DtiNet_Chemical_Protein.columns.str.lower()
print(f"DtiNet:    {len(DtiNet_Chemical_Protein):,} rows | columns: {list(DtiNet_Chemical_Protein.columns)}")

DtiNet_Chemical_Protein['kg_type'] = 'Generalised'
DtiNet_Chemical_Protein['species'] = 'Homo species'
DtiNet_Chemical_Protein.head(2)

DtiNet:    1,920 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'relation_source']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,relation_source,kg_type,species
0,25074887,ChemicalEntity_Protein,P22888,ChemicalEntity,Protein,DTINet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,C[C@H](C(=O)N)NC(=O)[C@@H]1CCCN1C(=O)[C@H](CCC...,Lutropin-choriogonadotropic hormone receptor,Pubchem,Uniprot_ID,DTINet,Generalised,Homo species
1,25074887,ChemicalEntity_Protein,P30968,ChemicalEntity,Protein,DTINet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,C[C@H](C(=O)N)NC(=O)[C@@H]1CCCN1C(=O)[C@H](CCC...,Gonadotropin-releasing hormone receptor,Pubchem,Uniprot_ID,DTINet,Generalised,Homo species


### 2d. STITCH

In [18]:
STITCH_Chemical_Protein = pd.read_csv(PROC_DIR + 'stitch/stitch_HUMAN_Chemical_Protein_STITCH.csv')
STITCH_Chemical_Protein.columns = STITCH_Chemical_Protein.columns.str.lower()
print(f"STITCH:    {len(STITCH_Chemical_Protein):,} rows | columns: {list(STITCH_Chemical_Protein.columns)}")

STITCH_Chemical_Protein['kg_type'] = 'Generalised'
STITCH_Chemical_Protein['species'] = 'Homo species'

STITCH_Chemical_Protein.head(2)

STITCH:    7,843,748 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smile_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,91668531,ChemicalEntity_Protein,P21917,ChemicalEntity,Protein,STITCH,"N-[4-(1-oxo-3,4-dihydro-2H-pyrido[4,3-b]indol-...",CC(=O)NCCCCN1C2=C(C3=CC=CC=C31)C(=O)NCC2,D(4) dopamine receptor,Pubchem,Uniprot_ID,Generalised,Homo species
1,91668531,ChemicalEntity_Protein,P46663,ChemicalEntity,Protein,STITCH,"N-[4-(1-oxo-3,4-dihydro-2H-pyrido[4,3-b]indol-...",CC(=O)NCCCCN1C2=C(C3=CC=CC=C31)C(=O)NCC2,B1 bradykinin receptor,Pubchem,Uniprot_ID,Generalised,Homo species


### 2e. biosnap  drugbank   bindingdb

In [19]:
biosnap = pd.read_csv(PROC_DIR + 'biosnap/Chemical_Protein_BioSNAP_Metaboglue.csv')
biosnap.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
biosnap.columns = biosnap.columns.str.lower()
biosnap['kg_type'] = 'Generalised'
biosnap['species'] = 'Homo species'
biosnap

,head,relation,tail,head_type,tail_type,kg_source,species,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP,Homo species,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID,Generalised
1,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP,Homo species,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID,Generalised
2,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP,Homo species,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID,Generalised
3,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP,Homo species,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID,Generalised
4,2145,ChemicalEntity_Protein,Q07217,ChemicalEntity,Protein,BioSNAP,Homo species,"3-(4-aminophenyl)-3-ethylpiperidine-2,6-dione",CCC1(CCC(=O)NC1=O)C2=CC=C(C=C2)N,"Cholesterol side-chain cleavage enzyme, mitoch...",Pubchem,Uniprot_ID,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9761,17429,ChemicalEntity_Protein,O97959,ChemicalEntity,Protein,BioSNAP,Homo species,piperidine-1-carbaldehyde,C1CCN(CC1)C=O,Alcohol dehydrogenase 1C,Pubchem,Uniprot_ID,Generalised
9762,131704274,ChemicalEntity_Protein,Q6C9Q0,ChemicalEntity,Protein,BioSNAP,Homo species,"3-[(1Z,4Z,8E,9Z,13Z,14Z)-18-(2-carboxyethyl)-8...",C/C=C\1/C(=C2/C=C\3/C(=C/C)/C(=C([N-]3)/C=C\4/...,Cytochrome c,Pubchem,Uniprot_ID,Generalised
9763,10915062,ChemicalEntity_Protein,Q9JJ10,ChemicalEntity,Protein,BioSNAP,Homo species,1-[2-(3-aminophenyl)-5-tert-butylpyrazol-3-yl]...,CC(C)(C)C1=NN(C(=C1)NC(=O)NC2=CC=CC=C2)C3=CC=C...,Proto-oncogene tyrosine-protein kinase Src,Pubchem,Uniprot_ID,Generalised
9764,11691442,ChemicalEntity_Protein,Q5EAN3,ChemicalEntity,Protein,BioSNAP,Homo species,8-bromo-4-(2-chlorophenyl)-N-(2-hydroxyethyl)-...,CN1C2=C(C3=C(C(=C2)C4=CC=CC=C4Cl)C(=O)NC3=O)C(...,Wee1-like protein kinase,Pubchem,Uniprot_ID,Generalised


In [20]:
drugbank = pd.read_csv(PROC_DIR + 'drugbank/Chemical_Protein_DrugBank_Metaboglue.csv')
drugbank.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
drugbank.columns = drugbank.columns.str.lower()

drugbank['kg_type'] = 'Generalised'
drugbank['species'] = 'Homo species'
drugbank['kg_source'] = 'Drugbank'

drugbank

,head,relation,tail,head_type,tail_type,kg_source,species,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,657181,ChemicalEntity_Protein,Q9TTI8,ChemicalEntity,Protein,Drugbank,Homo species,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=...,Gonadotropin-releasing hormone receptor,Pubchem,Uniprot_ID,Generalised
1,5311128,ChemicalEntity_Protein,Q28585,ChemicalEntity,Protein,Drugbank,Homo species,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...,Lutropin-choriogonadotropic hormone receptor,Pubchem,Uniprot_ID,Generalised
2,5311128,ChemicalEntity_Protein,Q9TTI8,ChemicalEntity,Protein,Drugbank,Homo species,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...,Gonadotropin-releasing hormone receptor,Pubchem,Uniprot_ID,Generalised
3,5311065,ChemicalEntity_Protein,Q53ZC4,ChemicalEntity,Protein,Drugbank,Homo species,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...,Vasopressin V2 receptor,Pubchem,Uniprot_ID,Generalised
4,5311065,ChemicalEntity_Protein,P48043,ChemicalEntity,Protein,Drugbank,Homo species,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...,Vasopressin V1a receptor,Pubchem,Uniprot_ID,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10134,137278711,ChemicalEntity_Protein,Q05147,ChemicalEntity,Protein,Drugbank,Homo species,6-fluoro-7-(2-fluoro-6-hydroxyphenyl)-1-(4-met...,C[C@H]1CN(CCN1C2=NC(=O)N(C3=NC(=C(C=C32)F)C4=C...,GTPase KRas,Pubchem,Uniprot_ID,Generalised
10135,64945,ChemicalEntity_Protein,P23175,ChemicalEntity,Protein,Drugbank,Homo species,"(1S,2R,4aS,6aR,6aS,6bR,8aR,10S,12aR,14bS)-10-h...",C[C@@H]1CC[C@@]2(CC[C@@]3(C(=CC[C@H]4[C@]3(CC[...,GTPase HRas,Pubchem,Uniprot_ID,Generalised
10136,64945,ChemicalEntity_Protein,Q61515,ChemicalEntity,Protein,Drugbank,Homo species,"(1S,2R,4aS,6aR,6aS,6bR,8aR,10S,12aR,14bS)-10-h...",C[C@@H]1CC[C@@]2(CC[C@@]3(C(=CC[C@H]4[C@]3(CC[...,Neutrophil elastase,Pubchem,Uniprot_ID,Generalised
10137,169535,ChemicalEntity_Protein,O35172,ChemicalEntity,Protein,Drugbank,Homo species,iron(3+);2-methyl-4-oxopyran-3-olate,CC1=C(C(=O)C=CO1)[O-].CC1=C(C(=O)C=CO1)[O-].CC...,Natural resistance-associated macrophage prote...,Pubchem,Uniprot_ID,Generalised


In [21]:
bindingdb = pd.read_csv(PROC_DIR + 'bindingdb/Chemical_Protein_Human_BindingDB.csv')
bindingdb.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
bindingdb.columns = bindingdb.columns.str.lower()
bindingdb['kg_type'] = 'Generalised'
bindingdb['species'] = 'Homo species'
bindingdb['kg_source'] = 'BindingDB'

bindingdb

,head,relation,tail,head_type,tail_type,kg_source,species,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,3009304,ChemicalEntity_Protein,P35968,ChemicalEntity,Protein,BindingDB,Homo species,"6-[(4R,5S,6S,7R)-4,7-dibenzyl-3-(5-carboxypent...",C1=CC=C(C=C1)C[C@@H]2[C@@H]([C@H]([C@H](N(C(=O...,Vascular endothelial growth factor receptor 2 ...,Pubchem,Uniprot_ID,Generalised
1,44520983,ChemicalEntity_Protein,P0DMS8,ChemicalEntity,Protein,BindingDB,Homo species,2-amino-N-[(1-ethylimidazol-2-yl)methyl]-6-(5-...,CCN1C=CN=C1CNC(=O)C2=NC(=NC(=C2)C3=CC=C(O3)C)N,Adenosine receptor A3,Pubchem,Uniprot_ID,Generalised
2,49831257,ChemicalEntity_Protein,P52333,ChemicalEntity,Protein,BindingDB,Homo species,"N-[5-[4-[(1,1-dioxo-1,4-thiazinan-4-yl)methyl]...",C1CC1C(=O)NC2=NN3C(=N2)C=CC=C3C4=CC=C(C=C4)CN5...,Tyrosine-protein kinase JAK3 {ECO:0000305},Pubchem,Uniprot_ID,Generalised
3,72712978,ChemicalEntity_Protein,P36888,ChemicalEntity,Protein,BindingDB,Homo species,8-(3-aminophenyl)-2-(4-methoxyanilino)pteridin...,COC1=CC=C(C=C1)NC2=NC=C3C(=N2)N(C(=O)C=N3)C4=C...,Receptor-type tyrosine-protein kinase FLT3,Pubchem,Uniprot_ID,Generalised
4,5327301,ChemicalEntity_Protein,P42575,ChemicalEntity,Protein,BindingDB,Homo species,5-[[[5-[[(2S)-1-carboxy-4-[(2-chlorophenyl)met...,CN(CC1=CC(=C(C=C1)O)C(=O)O)CC2=CC=C(S2)C(=O)N[...,Caspase-2 subunit p12,Pubchem,Uniprot_ID,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...
207848,145977670,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo species,"1-(4-methylsulfonylphenyl)-3-(1H-pyrrolo[2,3-b...",CS(=O)(=O)C1=CC=C(C=C1)N2C3=NC=NC(=C3C(=N2)C4=...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID,Generalised
207849,145977679,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo species,"6-[4-amino-3-(1H-pyrrolo[2,3-b]pyridin-5-yl)py...",C1CC2=C(CC1N3C4=NC=NC(=C4C(=N3)C5=CN=C6C(=C5)C...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID,Generalised
207850,118025908,ChemicalEntity_Protein,Q13535,ChemicalEntity,Protein,BindingDB,Homo species,"1-(2,3-dihydro-1H-inden-2-yl)-3-(1H-pyrrolo[2,...",C1C(CC2=CC=CC=C21)N3C4=NC=NC(=C4C(=N3)C5=CN=C6...,Serine/threonine-protein kinase ATR,Pubchem,Uniprot_ID,Generalised
207851,1363752,ChemicalEntity_Protein,Q99814,ChemicalEntity,Protein,BindingDB,Homo species,3-[[2-(4-tert-butylphenoxy)acetyl]amino]benzoi...,CC(C)(C)C1=CC=C(C=C1)OCC(=O)NC2=CC=CC(=C2)C(=O)O,Endothelial PAS domain-containing protein 1,Pubchem,Uniprot_ID,Generalised


## 3. Check and Fix Duplicate Columns

In [22]:
SOURCE_DFS = [
    ('CKG_Chemical_Protein_1',    CKG_Chemical_Protein_1),
    ('CKG_Chemical_Protein_2',    CKG_Chemical_Protein_2),
    ('CKG_Chemical_Protein_3',    CKG_Chemical_Protein_3),
    
    ('CrossBAR_Chemical_Protein_1', CrossBAR_Chemical_Protein_1),
    ('CrossBAR_Chemical_Protein_2', CrossBAR_Chemical_Protein_2),
    ('DtiNet_Chemical_Protein',   DtiNet_Chemical_Protein),
    ('STITCH_Chemical_Protein',   STITCH_Chemical_Protein),
    ('biosnap',               biosnap),
    ('drugbank',               drugbank),
    ('bindingdb',               bindingdb),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Chemical_Protein_1] ✓ no duplicates
[CKG_Chemical_Protein_2] ✓ no duplicates
[CKG_Chemical_Protein_3] ✓ no duplicates
[CrossBAR_Chemical_Protein_1] ✓ no duplicates
[CrossBAR_Chemical_Protein_2] ✓ no duplicates
[DtiNet_Chemical_Protein] ✓ no duplicates
[STITCH_Chemical_Protein] ✓ no duplicates
[biosnap] ✓ no duplicates
[drugbank] ✓ no duplicates
[bindingdb] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [23]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)

# Drop null / blank / 'nan' head rows
final_df = final_df[
    final_df['head'].notna() &
    (final_df['head'].astype(str).str.strip() != '') &
    (final_df['head'].astype(str).str.lower() != 'nan')
].reset_index(drop=True)
final_df['head'] = final_df['head'].astype(str)

print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 11,441,008 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,DB06439,ChemicalEntity_Protein,Q16283,ChemicalEntity,association,Protein,CKG,Drugbank,UniprotID,Tyloxapol,Lipoprotein lipase,Generalised,Homo species
1,DB06439,ChemicalEntity_Protein,B2R5T9,ChemicalEntity,association,Protein,CKG,Drugbank,UniprotID,Tyloxapol,Lipoprotein lipase,Generalised,Homo species


In [24]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,10608191,10608191,21216382
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,1232055,1232055,2464110


## 5. Standardise Fixed-Value Columns

In [25]:
final_df['head_type']  = 'ChemicalEntity'
final_df['relation']   = 'ChemicalEntity_Protein'
final_df['tail_type']  = 'Protein'
final_df['tail_id_is'] = 'Uniprot_ID'

# head_id_is: DB-prefix → Drugbank, else Pubchem
final_df['head_id_is'] = np.where(
    final_df['head'].isna(), None,
    np.where(final_df['head'].str.startswith('DB'), 'Drugbank', 'Pubchem')
)

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'ChemicalEntity_Protein'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'Protein'}
Unique kg_source: {'CKG', 'CROssBAR', 'BindingDB', 'BioSNAP', 'DTINet', 'Drugbank', 'STITCH'}


## 6. Deduplicate

Also filter out invalid `DBI`-prefixed head IDs after groupby.

In [26]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})

# Drop invalid DBI-prefixed head IDs (non-standard DrugBank artefacts)
final_df_dedup = final_df_dedup[~final_df_dedup['head'].str.startswith('DBI')].reset_index(drop=True)
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 7,163,619 rows


## 7. Resolve Head (Chemical) and Tail (Protein) Names

**Head:**
1. PubChem IUPAC lookup (numeric CIDs).
2. Normalise DrugBank IDs to zero-padded 5-digit format.
3. DrugBank name fallback for `DB`-prefixed IDs.
4. Drop rows where `head_detail_name` still unresolved.

**Tail:** UniProt RecName lookup; drop rows still missing `tail_detail_name`.

In [27]:
# ── Head ──────────────────────────────────────────────────────────────────────
final_df_dedup['head_detail_name'] = final_df_dedup['head'].astype(str).map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['head_smiles_name'] = final_df_dedup['head'].astype(str).map(Pubchem_CID_Smile_Dict)

def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['head'] = final_df_dedup['head'].apply(standardize_drugbank_id).astype(str)

mask = final_df_dedup['head_detail_name'].isna() & final_df_dedup['head'].str.startswith('DB')
final_df_dedup.loc[mask, 'head_detail_name'] = final_df_dedup.loc[mask, 'head'].map(Drugbank_dict)

before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} unresolvable head rows")

# ── Tail ──────────────────────────────────────────────────────────────────────
final_df_dedup['tail_detail_name'] = final_df_dedup['tail'].map(Uniprot_Recname_dict)

before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} unresolvable tail rows → {len(final_df_dedup):,} remaining")

Dropped 216,151 unresolvable head rows
Dropped 19,788 unresolvable tail rows → 6,927,680 remaining


## 8. Add Schema Columns and Enforce Column Order

In [28]:

EXTRA_COLS = ['head_smiles_name']
final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]

print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (6927680, 14)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,1000,ChemicalEntity_Protein,P11086,ChemicalEntity,None,Protein,CKG,Pubchem,Uniprot_ID,2-amino-1-phenylethanol,Phenylethanolamine N-methyltransferase,Generalised,Homo species,C1=CC=C(C=C1)C(CN)O
1,10000015,ChemicalEntity_Protein,P04350,ChemicalEntity,None,Protein,STITCH,Pubchem,Uniprot_ID,"N-[(9S)-5,19-dimethoxy-6-oxo-15,17-dioxatetrac...",Tubulin beta-4A chain,Generalised,Homo species,CC(=O)N[C@H]1CCC2=CC3=C(C(=C2C4=CC=C(C(=O)C=C1...
2,10000015,ChemicalEntity_Protein,P07437,ChemicalEntity,None,Protein,STITCH,Pubchem,Uniprot_ID,"N-[(9S)-5,19-dimethoxy-6-oxo-15,17-dioxatetrac...",Tubulin beta chain,Generalised,Homo species,CC(=O)N[C@H]1CCC2=CC3=C(C(=C2C4=CC=C(C(=O)C=C1...
3,10000015,ChemicalEntity_Protein,P68371,ChemicalEntity,None,Protein,STITCH,Pubchem,Uniprot_ID,"N-[(9S)-5,19-dimethoxy-6-oxo-15,17-dioxatetrac...",Tubulin beta-4B chain,Generalised,Homo species,CC(=O)N[C@H]1CCC2=CC3=C(C(=C2C4=CC=C(C(=O)C=C1...
4,10000015,ChemicalEntity_Protein,Q13509,ChemicalEntity,None,Protein,STITCH,Pubchem,Uniprot_ID,"N-[(9S)-5,19-dimethoxy-6-oxo-15,17-dioxatetrac...",Tubulin beta-3 chain,Generalised,Homo species,CC(=O)N[C@H]1CCC2=CC3=C(C(=C2C4=CC=C(C(=O)C=C1...


## 9. QC — NaN Check

In [29]:

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,6220855,0,6220855
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [30]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 6,927,680 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_PROTEIN/ALL_CHEMICALENTITY_PROTEIN.csv


In [1]:
#

